In [1]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.3 s3fs==2026.3.0 -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Import des packages nécessaires

In [138]:
import pandas as pd
import io
import os
import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation





# Import des données
Les données sont disponibles sur le site [DATA AMELI](https://data.ameli.fr/pages/home/).




J’ai choisi de travailler avec deux jeux de données totalement distincts :
**Une vue régionale et départementale**, décrivant les patients selon la pathologie, le traitement chronique ou l’épisode de soins, le sexe, la classe d’âge, la région et le département.  
  - Source : [Effectifs – Patients par pathologie](https://data.ameli.fr/explore/dataset/effectifs/information/)

**Une vue nationale**, portant sur les dépenses remboursées par pathologie et par patient en moyenne.  
  - Source : [Dépenses remboursées par pathologie](https://data.ameli.fr/explore/dataset/depenses/information/)

Comme le fichier **effectifs** est particulièrement volumineux, j’ai opté pour une **lecture en chunks**, ce qui permet de filtrer les données au fur et à mesure du chargement et d’éviter une surcharge mémoire.

J’ai ensuite enrichi ces données en les fusionnant avec un second fichier **region_dept** contenant les libellés des régions et des départements, afin d’obtenir un rendu plus harmonieux et plus lisible dans les visualisations.

Pour garantir un chargement fiable et éviter les problèmes liés aux chemins locaux, j’ai préféré déposer mes fichiers CSV sur Onyxia et les récupérer directement via leur URL MinIO. Cette approche assure un accès stable et reproductible aux données, quel que soit l’environnement d’exécution.

### Effectif de patients par pathologie, sexe, classe d'âge et territoire (département, région)

In [3]:
chunks = []

for chunk in pd.read_csv('https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/effectifs.csv', sep=';', chunksize=100_000,low_memory=False):
    filtered = chunk[(chunk['annee'] >= 2021) & (chunk['dept'] != '999')]
    chunks.append(filtered)

df_effectifs = pd.concat(chunks, ignore_index=True)

#Le datasaet qui va servir de jointure pour récuperer les libelles des départements et région
df_regions_dept = pd.read_csv(
    "https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/LibelleRegionDept.csv",
    sep=";",
    encoding="latin1",
    usecols=["departement", "libelle_departement", "libelle_region"]
)

# Harmonisation des colonnes , les 1 devient des 01 
df_regions_dept["departement"] = df_regions_dept["departement"].astype(str).str.zfill(2)
df_effectifs["dept"] = df_effectifs["dept"].astype(str).str.zfill(2)

# Jointure des 2 :
df_fusion = pd.merge(
    df_effectifs, 
    df_regions_dept, 
    left_on="dept", 
    right_on="departement", 
    how="inner"
)



Ici, je n’utilise pas `.copy()` car mon objectif n’est pas de dupliquer le DataFrame, mais simplement de le renommer.  
En faisant :

In [4]:
df_effectifs = df_fusion
del df_fusion # Et par la suite je supprime le dataFrame pour libérer de l'espace

## Nettoyage du dataset : df_effectifs

### Vérification et traitement des doublons

J’affiche ici les lignes du DataFrame qui apparaissent en doublon afin d’identifier
les répétitions éventuelles dans les données. L’option `keep=False` permet de visualiser
toutes les occurrences d’un doublon, ce qui facilite le diagnostic avant nettoyage.

Le fichier des correspondances *région–département* contient plusieurs doublons,ce qui génère des lignes dupliquées après la jointure.
Pour éviter ces répétitions dans le DataFrame final, j’ai décidé de supprimer les doublons avant de poursuivre l’analyse.


In [5]:
df_effectifs[df_effectifs.duplicated(keep=False)]


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
1,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
2,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
3,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
4,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7317445,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
7317446,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
7317447,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques
7317448,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,MCV_APE_IND,45-49,2,75,64,40.0,21590,0.185,"2,3",de 45 à 49 ans,femmes,27.0,Nouvelle-Aquitaine,64,Pyrénées-Atlantiques


#### Nombre de doublons par colonne

In [6]:
df_effectifs.apply(lambda col: col.duplicated().sum())


annee                  7317447
patho_niv1             7317432
patho_niv2             7317401
patho_niv3             7317388
top                    7317371
cla_age_5              7317429
sexe                   7317447
region                 7317432
dept                   7317349
Ntop                   7307633
Npop                   7310709
prev                   7266931
Niveau prioritaire     7317444
libelle_classe_age     7317429
libelle_sexe           7317447
tri                    7317371
libelle_region         7317432
departement            7317349
libelle_departement    7317349
dtype: int64

### Poids du dataset avant et après suppression des doublons

Avant de nettoyer les données, j’affiche le poids mémoire du DataFrame afin d’évaluer l’impact des doublons sur la taille totale du fichier.  
Après suppression des lignes dupliquées, j’affiche à nouveau le poids du dataset pour mesurer le gain en mémoire et vérifier que le nettoyage a bien été appliqué.


In [7]:
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids AVANT nettoyage :", buf.getvalue().splitlines()[-1])



Poids AVANT nettoyage : memory usage: 1.0 GB


In [8]:
# Suppression des doublons
df_effectifs = df_effectifs.drop_duplicates()


In [9]:
# Afficher uniquement la ligne "memory usage"
buf = io.StringIO()
df_effectifs.info(buf=buf, verbose=False)
print("Poids APRÈS nettoyage :", buf.getvalue().splitlines()[-1])

Poids APRÈS nettoyage : memory usage: 212.1 MB


In [10]:
df_effectifs.head()


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,libelle_region,departement,libelle_departement
0,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,8460,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,59,Nord
5,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,60,NaN,2470,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,60,Oise
10,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,62,NaN,5010,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,62,Pas-de-Calais
15,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,80,NaN,2220,NaN,3,plus de 95 ans,tous sexes,78.0,Hauts-de-France,80,Somme
20,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,44,10,NaN,1570,NaN,3,plus de 95 ans,tous sexes,78.0,Grand Est,10,Aube


In [11]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                       0
patho_niv1                  0
patho_niv2             152712
patho_niv3             330876
top                         0
cla_age_5                   0
sexe                        0
region                      0
dept                        0
Ntop                   390147
Npop                        0
prev                   390147
Niveau prioritaire      19089
libelle_classe_age          0
libelle_sexe                0
tri                     19089
libelle_region              0
departement                 0
libelle_departement         0
dtype: int64

Dans ce jeu de données, les valeurs indiquées comme NaN ne correspondent pas à de véritables valeurs manquantes. Elles apparaissent simplement lorsqu’un département ne présente aucun cas pour une pathologie donnée. Dans ce contexte, il est donc logique de remplacer ces NaN par 0, afin de refléter correctement l’absence de cas observés.

In [12]:
df_effectifs = df_effectifs.fillna(0)


###  Après avoir passer les NAN à 0

In [13]:
# Valeurs manquantes
(
    df_effectifs
    .isna()
    .sum(axis=0)
)

annee                  0
patho_niv1             0
patho_niv2             0
patho_niv3             0
top                    0
cla_age_5              0
sexe                   0
region                 0
dept                   0
Ntop                   0
Npop                   0
prev                   0
Niveau prioritaire     0
libelle_classe_age     0
libelle_sexe           0
tri                    0
libelle_region         0
departement            0
libelle_departement    0
dtype: int64

### Nettoyage et préparation des données

- J’ai décidé de supprimer les parenthèses dans les variables *patho_niv2* et *patho_niv3*, car elles entraînaient un affichage peu lisible dans les visualisations et ajoutaient trop d’informations inutiles.
- Je me suis concentrée sur la France hexagonale, en incluant la Corse et les DOM.
- Les codes **FRANCE** et **Tout département** correspondent à des agrégats régionaux ou nationaux (valeurs non rattachées à un département).  
  Comme ils faussent l’analyse territoriale, j’ai choisi de les supprimer.


In [14]:
# Nettoyage des parenthèses
cols = ["patho_niv2", "patho_niv3"]
for c in cols:
    df_effectifs.loc[:, c] = df_effectifs[c].str.replace(r"\s*\(.*\)", "", regex=True)

# Grand filtre de nettoyage , je veux gardé que la france hexagonale
hors_hexa = ['Tout département', 'Guadeloupe ', 'Haute-Corse', 'Martinique', 'La Réunion', 'Guyane', 'Mayotte', 'Corse-du-Sud','FRANCE']

df_effectifs = df_effectifs[
    (~df_effectifs['libelle_departement'].astype(str).isin(hors_hexa)) &
    
    # On enlève la région "France" qui englobe tout
    (df_effectifs['libelle_region'].astype(str) != 'FRANCE')
]


Pour ne conserver que les observations réellement exploitables, je filtre le dataset
en gardant uniquement les lignes où la variable `prev` est strictement supérieure à 0.
Cela permet d’éliminer les enregistrements sans prévalence et d’alléger les analyses
et visualisations suivantes.

In [15]:
df_effectifs = df_effectifs[df_effectifs["prev"] > 0]


In [16]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2              object
patho_niv3              object
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

### Nettoyage des colonnes `patho_niv2` et `patho_niv3` après remplacement des `NaN`

Lors du remplacement des valeurs manquantes (`NaN`) par `0`, les colonnes `patho_niv2` et
`patho_niv3` ont changé de type : elles sont passées du type *string* au type *object*.
Lorsqu’une colonne contient un mélange de valeurs c'est ce que pandas fait numériques, textuelles ou des `NaN`, elle est automatiquement convertie en `object`.
Comme les `NaN` avaient été remplacés par `0`, ces valeurs sont restées sous forme de `"0"` après conversion en `str`.  
Je les ai supprimé pour ne conserver que les niveaux de pathologie valides.


In [17]:
df_effectifs["patho_niv2"] = df_effectifs["patho_niv2"].astype(str).str.strip()
df_effectifs["patho_niv3"] = df_effectifs["patho_niv3"].astype(str).str.strip()


In [18]:
df_effectifs.dtypes


annee                    int64
patho_niv1                 str
patho_niv2                 str
patho_niv3                 str
top                        str
cla_age_5                  str
sexe                     int64
region                   int64
dept                       str
Ntop                   float64
Npop                     int64
prev                   float64
Niveau prioritaire      object
libelle_classe_age         str
libelle_sexe               str
tri                    float64
libelle_region             str
departement                str
libelle_departement        str
dtype: object

In [19]:
df_effectifs = df_effectifs[
    (df_effectifs["patho_niv1"] != "0") &
    (df_effectifs["patho_niv2"] != "0") &
    (df_effectifs["patho_niv3"] != "0")
]

### Suppression des colonnes non utilisées

Dans cette étape, je retire les colonnes qui ne sont pas nécessaires pour l’analyse finale.  
Il s’agit principalement de variables techniques, de codes intermédiaires ou de niveaux d’agrégation
qui ne sont pas exploités dans les visualisations (ex. : codes de tri, niveaux prioritaires, variables
de regroupement, ou encore des informations redondantes comme la colonne **region** issue d’un des
datasets utilisés pour la jointure).

L’option `errors='ignore'` permet d’éviter une erreur si certaines colonnes ont déjà été supprimées
lors d’étapes précédentes du nettoyage.


In [20]:
# Suppression des colonnes inutiles pour le nettoyage final
df_effectifs = df_effectifs.drop(
    columns=[
        "Code département",
        "tri",
        "Niveau prioritaire",
        "region",
        "dept",
        "departement",
        "cla_age_5",
        "top",
        "sexe",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




### Renommage de certaines colonnes

Pour améliorer la lisibilité du dataset et faciliter la compréhension des analyses,
j’ai choisi de renommer plusieurs colonnes.  
Afin d’obtenir des intitulés plus courts, plus explicites et cohérents avec les visualisations produites par la suite.



In [21]:
# Renommage des colonens
df_effectifs = df_effectifs.rename(columns={
    "libelle_departement": "Département",
    "libelle_region": "Région",
    "prev" : "Prévalence",
    "Npop": "Population de référence",
    "Ntop": "Effectif",
    "libelle_sexe" : "Sexe",
    "libelle_classe_age" : "Classe d'age"


})

In [22]:
df_effectifs = df_effectifs.dropna()

In [23]:
df_effectifs

,annee,patho_niv1,patho_niv2,patho_niv3,Effectif,Population de référence,Prévalence,Classe d'age,Sexe,Région,Département
150,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,150.0,981490,0.015,tous âges,hommes,Ile-de-France,Paris
155,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,30.0,204050,0.015,tous âges,hommes,Centre-Val de Loire,Eure-et-Loir
160,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,20.0,123680,0.016,tous âges,hommes,Bourgogne-Franche-Comté,Jura
170,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,40.0,259740,0.014,tous âges,hommes,Bourgogne-Franche-Comté,Saône-et-Loire
175,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,10.0,65260,0.018,tous âges,hommes,Bourgogne-Franche-Comté,Territoire de Belfort
...,...,...,...,...,...,...,...,...,...,...,...
7317415,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,47150,0.091,de 45 à 49 ans,femmes,Pays de la Loire,Loire-Atlantique
7317420,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,21550,0.162,de 45 à 49 ans,femmes,Pays de la Loire,Vendée
7317425,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,30.0,17860,0.174,de 45 à 49 ans,femmes,Bretagne,Côtes-d'Armor
7317430,2023,Maladies cardioneurovasculaires,Artériopathie périphérique,Artériopathie périphérique,40.0,34710,0.107,de 45 à 49 ans,femmes,Bretagne,Ille-et-Vilaine


### Exploration des niveaux de pathologies

Avant d’appliquer des filtres plus précis, j’affiche les valeurs uniques de `patho_niv2` et `patho_niv3`
(en excluant les valeurs `NaN`). Cela permet de vérifier la cohérence des libellés et d’identifier
éventuels regroupements ou corrections à effectuer avant l’analyse.


In [24]:
print(df_effectifs['patho_niv1'].dropna().unique())
print(df_effectifs['patho_niv2'].dropna().unique())
print(df_effectifs['patho_niv3'].dropna().unique())


<StringArray>
[                                                               'Maladies inflammatoires ou rares ou infection VIH',
                                                                                           'Maladies neurologiques',
                                                                                          'Maladies psychiatriques',
 'Pas de pathologie repérée, traitement, maternité, hospitalisation ou traitement antalgique ou anti-inflammatoire',
                                                                                   'Total consommants tous régimes',
                                                              'Traitements du risque vasculaire (hors pathologies)',
                                                                      'Traitements psychotropes (hors pathologies)',
                                                                                  'Maladies cardioneurovasculaires',
                                                  

# 2 ème dataset que je vais étudier :

# Dépenses remboursées affectées à chaque pathologie

Les données présentent des informations sur les dépenses remboursées par l’ensemble des régimes d’assurance maladie et affectées à chaque pathologie, traitement chronique ou épisode de soins. Ces dépenses sont réparties en trois grandes catégories :

* les soins de ville (consultations médicales, soins infirmiers, actes de kinésithérapie, médicaments, biologie, transports, etc.)  ; 

* les hospitalisations dans les établissements de santé publics ou privés ;

* les prestations en espèces, notamment les indemnités journalières.

## Deux types d’indicateurs sont proposés pour chacune de ces catégories : 

* les dépenses totales annuelles remboursées, qui mesurent le poids financier global d’une pathologie ;

* les dépenses moyennes annuelles remboursées par patient, qui permettent d’apprécier le coût moyen associé à chaque pathologie.

Ces deux indicateurs offrent ainsi une vision complémentaire des dépenses de santé, en combinant à la fois l’impact global et la charge moyenne par patient.

In [25]:
chunks = []

for chunk in pd.read_csv(
    'https://minio.lab.sspcloud.fr/aissataa/PROJET_MEYTE/depenses.csv',
    sep=';',
    chunksize=100_000,
    low_memory=False
):
    filtered = chunk[
        (chunk['annee'] >= 2021) &
        (chunk['montant'] > 0) 
    ]
    chunks.append(filtered)

df_depenses = pd.concat(chunks, ignore_index=True)
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,top,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy,Niveau prioritaire,tri,type_somme
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0,"1,2,3",16.0,Partiel
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0,"1,2,3",16.0,Total
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0,"1,2,3",16.0,Partiel
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0,"1,2,3",16.0,Partiel
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0,"1,2,3",16.0,Partiel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6401,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0,"2,3",55.0,Partiel
6402,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0,"2,3",55.0,Partiel
6403,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0,"2,3",55.0,Partiel
6404,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),TPS_NLE_EXC,Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0,"2,3",55.0,Total


# Les champs sont les suivants avant nettoyage:

In [26]:
df_depenses.columns


Index(['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'dep_niv_1',
       'dep_niv_2', 'montant', 'Ntop', 'N_recourant_au_poste', 'montant_moy',
       'Niveau prioritaire', 'tri', 'type_somme'],
      dtype='str')

In [27]:
# Suppression des colonnes inutiles pour le nettoyage final
df_depenses = df_depenses.drop(
    columns=[
        "tri",
        "Niveau prioritaire",
        "type_somme",
        "top",

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)




# Nettoyage des données

In [28]:
(
    df_depenses
    .isna()
    .sum(axis=0)
)

annee                      0
patho_niv1                 0
patho_niv2               686
patho_niv3              1496
dep_niv_1                  0
dep_niv_2                  0
montant                    0
Ntop                      42
N_recourant_au_poste      42
montant_moy               42
dtype: int64

In [29]:
df_depenses = df_depenses.drop_duplicates()


In [30]:
df_depenses


,annee,patho_niv1,patho_niv2,patho_niv3,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy
0,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur priv...,3856836,1714310.0,34840.0,2.0
1,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Hospitalisations (tous secteurs),Total hospitalisations (tous secteurs) rembour...,308350458,1714310.0,1232630.0,180.0
2,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,117367790,1714310.0,125760.0,68.0
3,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Prestations en espèces,Prestations d'invalidité remboursées,263517389,1714310.0,68580.0,154.0
4,2023,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Soins de ville,Autres dépenses de soins de ville remboursés,9004352,1714310.0,424990.0,5.0
...,...,...,...,...,...,...,...,...,...,...
6401,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Autres produits de santé remboursés,10878069,335640.0,212200.0,32.0
6402,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Soins de généralistes remboursés,11874138,335640.0,300910.0,35.0
6403,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Soins dentaires remboursés,6030821,335640.0,118450.0,18.0
6404,2022,Traitements psychotropes (hors pathologies),Traitements neuroleptiques (hors pathologies),Traitements neuroleptiques (hors pathologies),Soins de ville,Total soins de ville remboursés,199465569,335640.0,335630.0,594.0


# Filtrage avant creation du dashboard

In [131]:
df_depenses = df_depenses.drop('Total consommants tous régimes', errors='ignore')

# Création du fichier Excel

Je vais créer un nouveau fichier Excel qui contiendra les données nettoyées dans un onglet "cleaned_data".

In [139]:
output_path = '../DATA/Dashboards1.xlsx'

In [140]:
with pd.ExcelWriter(output_path) as writer:
    df_depenses.to_excel(writer, sheet_name='cleaned_data', index=False)

In [141]:
from openpyxl.worksheet.formula import ArrayFormula

# Définition des colonnes à calculer
cols_to_calculate = ['annee', 'patho_niv1', 'dep_niv_1','dep_niv_2']
len_dict = {}

for col in cols_to_calculate:
    if col == 'patho_niv1':
        # On exclut les totaux pour avoir la vraie taille propre à afficher
        uniques = df_depenses.loc[
            ~df_depenses[col].isin(['Total', 'Total consommants tous régimes']), col
        ].dropna().unique()
    else:
        uniques = df_depenses[col].dropna().unique()
        
    # On rajoute +1 pour la ligne d'en-tête
    len_dict[f"len_{col}"] = len(uniques) + 1

len_dict

{'len_annee': 4, 'len_patho_niv1': 19, 'len_dep_niv_1': 5, 'len_dep_niv_2': 32}

Création de l'onglet Filtre et l'alimenter

In [143]:
path_results = '../DATA/Dashboards1.xlsx'
wb = load_workbook(path_results)

if 'Filtres' not in wb.sheetnames:
    # Si la feuille n'existe pas, la créer
    ress_sheet = wb.create_sheet('Filtres')
else:
    ress_sheet = wb['Filtres']

In [144]:
formula = "=_xlfn.SORT(_xlfn.UNIQUE(cleaned_data!A:A))"
ress_sheet['A1'] = ArrayFormula(f"A1:A{len_dict['len_annee']}", formula)

# Pathologies -> Colonne B dans cleaned_data
formula = "=_xlfn.SORT(_xlfn.UNIQUE(cleaned_data!B:B))"
ress_sheet['B1'] = ArrayFormula(f"B1:B{len_dict['len_patho_niv1']}", formula)

# Grands Postes (dep_niv_1) -> Colonne E dans cleaned_data
formula = "=_xlfn.UNIQUE(cleaned_data!E:E)"
ress_sheet['C1'] = ArrayFormula(f"C1:C{len_dict['len_dep_niv_1']}", formula)

# Postes Détaillés (dep_niv_2) -> Colonne F dans cleaned_data
formula = "=_xlfn.UNIQUE(cleaned_data!F:F)"
ress_sheet['D1'] = ArrayFormula(f"D1:D{len_dict['len_dep_niv_2']}", formula)

# Ajustement de la largeur des colonnes de A à K
for col_idx in range(1, 12):
    col_letter = ress_sheet.cell(row=1, column=col_idx).column_letter
    ress_sheet.column_dimensions[col_letter].width = 45

wb.save(path_results)
wb.close()

Création du dashboard

Je vais créer un premier tableau de bord interactif avec des filtres.
Je vais me concentrer sur les dépenses de remboursement par pathologie et par poste de dépense.
L'utilisateur aura la possibilité de choisir l'année, la pathologie et le poste de dépense via des filtres sous forme de listes de validation.
Ces filtres permettront d'actualiser dynamiquement les formules et les graphiques.

In [145]:
from openpyxl import load_workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import PatternFill, Alignment, Font, Border, Side

wb = load_workbook(path_results)

# --- 1. Gestion et initialisation de l'onglet 'Depenses' ---
if 'Depenses' not in wb.sheetnames:
    Depenses_sheet = wb.create_sheet('Depenses')
else:
    Depenses_sheet = wb['Depenses']

# On masque le quadrillage par défaut pour un rendu Dashboard propre
Depenses_sheet.sheet_view.showGridLines = False

# Configuration des styles graphiques
COULEURS = {
    'bleu_fonce': '1F4E78',
    'jaune_filtre': 'FFF2CC',
    'gris_clair': 'CCCCCC'
}
trait = Side(style='thin', color=COULEURS['gris_clair'])
bordure = Border(left=trait, right=trait, top=trait, bottom=trait)

# --- 2. Création personnalisée du filtre 'Année' ---

# Libellé du filtre (En-tête)
filter_1_cell = Depenses_sheet['A1']
filter_1_cell.value = 'Année'
filter_1_cell.font = Font(bold=True, color='FFFFFF', size=11)
filter_1_cell.alignment = Alignment(horizontal='center', vertical='center')
filter_1_cell.fill = PatternFill(start_color=COULEURS['bleu_fonce'], end_color=COULEURS['bleu_fonce'], fill_type='solid')

# Fusionner les cellules A1:B2 pour le titre du filtre
Depenses_sheet.merge_cells(start_row=1, start_column=1, end_row=2, end_column=2)

# Valeur sélectionnable du filtre (Case cliquable)
val_filter_1_cell = Depenses_sheet['C1']
val_filter_1_cell.value = 2023  # Année par défaut pour tes données Ameli
val_filter_1_cell.font = Font(bold=True, color='000000', size=11)
val_filter_1_cell.alignment = Alignment(horizontal='center', vertical='center')
val_filter_1_cell.fill = PatternFill(start_color=COULEURS['jaune_filtre'], end_color=COULEURS['jaune_filtre'], fill_type='solid')

# Application des bordures sur toute la zone fusionnée C1:D2
for r in range(1, 3):
    for c in range(3, 5):
        Depenses_sheet.cell(row=r, column=c).border = bordure

# Fusionner les cellules C1:D2 pour accueillir la valeur et la flèche de liste
Depenses_sheet.merge_cells(start_row=1, start_column=3, end_row=2, end_column=4)

# --- 3. Liaison avec la liste unique de l'onglet 'Filtres' ---
# La formule pointe sur la colonne A de ton onglet 'Filtres' (à partir de la ligne 2 pour éviter l'en-tête)
formula = f"='Filtres'!$A$2:$A${len_dict['len_annee']}"

# Injection de la validation de données dans la cellule C1
dv = DataValidation(type='list', formula1=formula)
Depenses_sheet.add_data_validation(dv)
coord_filter_year = 'C1'
dv.add(coord_filter_year)

# Sauvegarde des modifications
wb.save(path_results)
wb.close()

print("✅ Onglet 'Depenses' initialisé et filtre personnalisé 'Année' créé avec succès !")


✅ Onglet 'Depenses' initialisé et filtre personnalisé 'Année' créé avec succès !


Design :

In [146]:
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import PatternFill, Alignment, Font, Border, Side

COULEURS = {
    'bleu_fonce': '1F4E78',
    'jaune_filtre': 'FFF2CC',
    'gris_clair': 'CCCCCC'
}
trait = Side(style='thin', color=COULEURS['gris_clair'])
bordure = Border(left=trait, right=trait, top=trait, bottom=trait)

Comme nous allons répéter cette opération plusieurs fois, il est préférable de créer une fonction qui effectue les mêmes opérations.

In [147]:
def add_filter(
    sheet: Worksheet,
    title: str,
    value: str,
    start_row: int,
    start_column: int,
    end_row: int,
    end_column: int,
    formula: str) -> None:
    """
    Ajoute dun composant de filtre unique (Libellé + Liste déroulante) sur le Dashboard.
    
    En respectant le principe DRY et SRP, cette fonction orchestre l'appel 
    des fonctions filles pour créer le titre, la case jaune et la validation.
    """
    # 1. Je génère le libellé du filtre (ex: 'Filtrer par poste de dépenses')
    add_filter_title(sheet, title, start_row, start_column, end_row, end_column)

    # 2. Je génère la case de sélection jaune (placée juste à droite de l'en-tête)
    add_filter_value(sheet, value, start_row, end_column + 1, end_row, end_column + 2)

    # 3. J'injecte la liste déroulante Excel issue de l'onglet 'Filtres'
    add_data_validation(sheet, start_row, end_column + 1, formula)


def add_filter_title(
    sheet: Worksheet,
    title: str,
    start_row: int,
    start_column: int,
    end_row: int,
    end_column: int) -> None:
    """
    Crée et stylise la cellule de titre d'un filtre (Style Bleu Foncé).
    """
    filter_cell = sheet.cell(row=start_row, column=start_column)
    filter_cell.value = title
    filter_cell.font = Font(bold=True, color='FFFFFF', size=11)
    filter_cell.alignment = Alignment(horizontal='center', vertical='center')
    filter_cell.fill = PatternFill(start_color=COULEURS['bleu_fonce'], end_color=COULEURS['bleu_fonce'], fill_type='solid')
    
    sheet.merge_cells(start_row=start_row, start_column=start_column, end_row=end_row, end_column=end_column)


def add_filter_value(
    sheet: Worksheet,
    value: str,
    start_row: int,
    start_column: int,
    end_row: int,
    end_column: int) -> None:
    """
    Crée la zone cliquable du filtre (Style Jaune Clair) avec ses bordures fines.
    """
    # J'applique d'abord les bordures sur toutes les cellules de la zone avant la fusion
    for r in range(start_row, end_row + 1):
        for c in range(start_column, end_column + 1):
            sheet.cell(row=r, column=c).border = bordure

    val_filter_cell = sheet.cell(row=start_row, column=start_column)
    val_filter_cell.value = value
    val_filter_cell.font = Font(bold=True, color='000000', size=11)
    val_filter_cell.alignment = Alignment(horizontal='center', vertical='center')
    val_filter_cell.fill = PatternFill(start_color=COULEURS['jaune_filtre'], end_color=COULEURS['jaune_filtre'], fill_type='solid')
    
    sheet.merge_cells(start_row=start_row, start_column=start_column, end_row=end_row, end_column=end_column)


def add_data_validation(
    sheet: Worksheet,
    start_row: int,
    start_column: int,
    formula: str) -> None:
    """
    Lie une validation de données (liste déroulante) à la cellule Excel cible.
    """
    dv = DataValidation(type='list', formula1=formula)
    sheet.add_data_validation(dv)
    target_cell = sheet.cell(row=start_row, column=start_column)
    dv.add(target_cell.coordinate)

Bordure des filtres :

In [148]:
def apply_border_style(
    sheet: Worksheet,
    start_row: int, 
    end_row: int, 
    start_col: int, 
    end_col: int, 
    border_row: int) -> None:
    """
    Applique une bordure inférieure fine à une ligne spécifique dans une plage Excel.
    
    Je parcours la plage définie pour ajouter une bordure grise sur la ligne choisie,
    et je nettoie les bordures sur le reste de la plage pour garder un design épuré.
    """
    # J'utilise le gris clair de ma charte graphique pour la cohérence visuelle
    thin = Side(border_style="thin", color="CCCCCC")

    # Je parcours toutes les cellules de la zone délimitée
    for row in sheet.iter_rows(min_row=start_row, max_row=end_row, min_col=start_col, max_col=end_col):
        for cell in row:
            if cell.row == border_row:
                cell.border = Border(bottom=thin)
            else:
                cell.border = None

filtre

In [149]:
wb = load_workbook(path_results)

# On utilise ton onglet 'Depenses'
Depenses_sheet = wb['Depenses']

# Filtre 1 : Année (Ligne 1 à 2, Colonnes 1 à 2 -> Case de choix en C1/D2)
add_filter(
    sheet=Depenses_sheet,
    title='Année',
    value='2023',
    start_row=1,
    start_column=1,
    end_row=2,
    end_column=2,
    formula=f"='Filtres'!$A$2:$A${len_dict['len_annee']}"
)

# Filtre 2 : Poste de dépenses agrégé (Ligne 1 à 2, Colonnes 5 à 6 -> Case de choix en G1/H2)
add_filter(
    sheet=Depenses_sheet,
    title='Filtrer par poste de dépenses',
    value='Soins de ville',
    start_row=1,
    start_column=5,
    end_row=2,
    end_column=6,
    formula=f"='Filtres'!$C$2:$C${len_dict['len_dep_niv_1']}"
)

# Filtre 3 : Pathologie (Ligne 1 à 2, Colonnes 9 à 10 -> Case de choix en K1/L2)
add_filter(
    sheet=Depenses_sheet,
    title='Pathologie',
    value='Toutes pathologies',
    start_row=1,
    start_column=9,
    end_row=2,
    end_column=10,
    formula=f"='Filtres'!$B$2:$B${len_dict['len_patho_niv1']}"
)

# Application de la bordure inférieure grise pour fermer proprement le bloc des filtres
apply_border_style(Depenses_sheet, start_row=1, end_row=2, start_col=1, end_col=12, border_row=2)

# Sauvegarder et fermer le fichier
wb.save(path_results)
wb.close()

In [ ]:

# Je redimensionne les colonnes de A à L pour éviter que les textes longs ou les montants affichent des '###'
for col_idx in range(1, 13):
    col_letter = get_column_letter(col_idx)
    Depenses_sheet.column_dimensions[col_letter].width = 10

# Je sauvegarde mes modifications et je ferme proprement le classeur
wb.save(path_results)
wb.close()

print(" Dashboard 'Depenses' entièrement généré, mis en forme et sauvegardé !")

✅ Dashboard 'Depenses' entièrement généré, mis en forme et sauvegardé !


In [151]:
wb = load_workbook(path_results)

if 'calc_sheet' not in wb.sheetnames:
    calc_sheet = wb.create_sheet('calc_sheet')
else:
    calc_sheet = wb['calc_sheet']

calc_sheet['A1'] = "Année"
calc_sheet['B1'] = "Nb Patients"
calc_sheet['C1'] = "Coût Moyen par Patient"

for i in range(2, len_dict['len_annee'] + 2):
    calc_sheet.cell(row=i, column=1, value=f"=Filtres!A{i}")
    calc_sheet.cell(row=i, column=2, value=f"=SUMIFS(cleaned_data!H:H, cleaned_data!A:A, A{i})")
    calc_sheet.cell(row=i, column=3, value=f"=IFERROR(SUMIFS(cleaned_data!G:G, cleaned_data!A:A, A{i}) / B{i}, 0)")

calc_sheet['E1'] = "Pathologie"
calc_sheet['F1'] = "Nb Patients Patho"
calc_sheet['G1'] = "Coût Moyen Patho"

for i in range(2, len_dict['len_patho_niv1'] + 2):
    calc_sheet.cell(row=i, column=5, value=f"=Filtres!B{i}")
    calc_sheet.cell(row=i, column=6, value=f"=SUMIFS(cleaned_data!H:H, cleaned_data!B:B, E{i})")
    calc_sheet.cell(row=i, column=7, value=f"=IFERROR(SUMIFS(cleaned_data!G:G, cleaned_data!B:B, E{i}) / F{i}, 0)")

calc_sheet['I1'] = "Pathologie (Dépenses)"
calc_sheet['J1'] = "Montant Total"

for i in range(2, len_dict['len_patho_niv1'] + 2):
    calc_sheet.cell(row=i, column=9, value=f"=Filtres!B{i}")
    calc_sheet.cell(row=i, column=10, value=f"=SUMIFS(cleaned_data!G:G, cleaned_data!B:B, I{i})")

calc_sheet['L1'] = "Poste dep_niv1"
calc_sheet['M1'] = "Montant Total"

for i in range(2, len_dict['len_dep_niv_1'] + 2):
    calc_sheet.cell(row=i, column=12, value=f"=Filtres!C{i}")
    calc_sheet.cell(row=i, column=13, value=f"=SUMIFS(cleaned_data!G:G, cleaned_data!E:E, L{i})")

wb.save(path_results)
wb.close()

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import PatternFill, Alignment, Font, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
import time

output = '../DATA/Dashboard1.xlsx'

with pd.ExcelWriter(output, engine='openpyxl') as w:
    df_depenses.to_excel(w, sheet_name='cleaned_data', index=False)

time.sleep(0.5)
wb = load_workbook(output)
ws_data = wb['cleaned_data']

# COLONNES
c_annee    = 'A'
c_patho    = 'B'
c_dep_niv1 = 'E'
c_dep_niv2 = 'F'
c_montant  = 'G'
c_ntop     = 'H'
c_moy      = 'J'

COULEURS = {
    'bleu_fonce': '1F4E78', 'bleu': '2F5496', 'bleu_clair': '4472C4',
    'vert': '70AD47', 'orange': 'C65911', 'jaune': 'FFF2CC', 'rose': 'D946A6',
    'gris_clair': 'CCCCCC'
}
trait = Side(style='thin', color='CCCCCC')
bordure = Border(left=trait, right=trait, top=trait, bottom=trait)

def entete(cell, texte, couleur):
    cell.value = texte
    cell.font = Font(bold=True, size=11, color='FFFFFF')
    cell.fill = PatternFill(start_color=couleur, end_color=couleur, fill_type='solid')
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border = bordure

annees = sorted(df_depenses['annee'].dropna().astype(int).unique())
grand_postes = sorted(df_depenses['dep_niv_1'].dropna().unique())
pathos = sorted([p for p in df_depenses['patho_niv1'].unique() 
                 if str(p).strip() not in ['Total', 'Total consommants tous régimes']])

cols_to_calculate = ['annee', 'dep_niv_1', 'dep_niv_2', 'patho_niv1']
len_dict = {}
for col in cols_to_calculate:
    if col == 'annee':
        len_dict[f"len_{col}"] = len(annees) + 1
    elif col == 'dep_niv_1':
        len_dict[f"len_{col}"] = len(grand_postes) + 1

    elif col == 'patho_niv1':
        len_dict[f"len_{col}"] = len(pathos) + 1

if 'Filtres' in wb.sheetnames: 
    del wb['Filtres']
ress = wb.create_sheet('Filtres', 0)

ress['A1'] = 'Annee'
for i, a in enumerate(sorted(annees), 2):
    ress[f'A{i}'] = int(a)

ress['B1'] = 'Grand Postes'
for i, gp in enumerate(grand_postes, 2):
    ress[f'B{i}'] = gp

ress['D1'] = 'Pathologies'
for i, p in enumerate(pathos, 2):
    ress[f'D{i}'] = p

for col in ['A', 'B', 'D']:
    ress.column_dimensions[col].width = 45

ress.sheet_state = 'hidden'

if 'Dashboard' in wb.sheetnames: 
    del wb['Dashboard']
ws = wb.create_sheet('Dashboard', 1)
ws.sheet_view.showGridLines = False

ws['A1'] = "CARTOGRAPHIE DES DÉPENSES"
ws['A1'].font = Font(bold=True, size=16, color='FFFFFF')
ws['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], end_color=COULEURS['bleu_fonce'], fill_type='solid')
ws.merge_cells('A1:F1')
ws.row_dimensions[1].height = 35

ws['A3'] = 'Année :'
ws['A3'].font = Font(bold=True, size=11)
ws['B3'].fill = PatternFill(start_color=COULEURS['jaune'], end_color=COULEURS['jaune'], fill_type='solid')
ws['B3'].border = bordure
dv_a = DataValidation(type='list', formula1=f"=Ressources!$A$2:$A${len_dict['len_annee']}")
ws.add_data_validation(dv_a)
dv_a.add('B3')
ws['B3'] = int(annees[-1])

ws['C3'] = 'Grand Poste :'
ws['C3'].font = Font(bold=True, size=11)
ws['D3'].fill = PatternFill(start_color=COULEURS['jaune'], end_color=COULEURS['jaune'], fill_type='solid')
ws['D3'].border = bordure
dv_gp = DataValidation(type='list', formula1=f"=Ressources!$B$2:$B${len_dict['len_dep_niv_1']}")
ws.add_data_validation(dv_gp)
dv_gp.add('D3')
ws['D3'] = grand_postes[0]

dv_pd = DataValidation(type='list', formula1=f"=Ressources!$C$2:$C${len_dict['len_dep_niv_2']}")
ws.add_data_validation(dv_pd)
dv_pd.add('F3')
ws['F3'] = postes_detail[0] if postes_detail else ""



 Longueurs calculées :
  len_annee    = 4
  len_dep_niv_1   = 5
  len_dep_niv_2   = 32
  len_patho_niv1  = 19

 4 grands postes | 31 postes | 18 pathologies | 3 années
Dashboard


In [ ]:
formula = "=_xlfn.SORT(_xlfn.UNIQUE(cleaned_data!A:A))"
ress_sheet['A1'] = ArrayFormula(f"A1:A{len_dict['len_annee']}", formula)

# Pathologies -> Colonne B dans cleaned_data
formula = "=_xlfn.SORT(_xlfn.UNIQUE(cleaned_data!B:B))"
ress_sheet['B1'] = ArrayFormula(f"B1:B{len_dict['len_patho_niv1']}", formula)

# Grands Postes (dep_niv_1) -> Colonne E dans cleaned_data
formula = "=_xlfn.UNIQUE(cleaned_data!E:E)"
ress_sheet['C1'] = ArrayFormula(f"C1:C{len_dict['len_dep_niv_1']}", formula)

# Postes Détaillés (dep_niv_2) -> Colonne F dans cleaned_data
formula = "=_xlfn.UNIQUE(cleaned_data!F:F)"
ress_sheet['D1'] = ArrayFormula(f"D1:D{len_dict['len_dep_niv_2']}", formula)

# Ajustement de la largeur des colonnes de A à K
for col_idx in range(1, 12):
    col_letter = ress_sheet.cell(row=1, column=col_idx).column_letter
    ress_sheet.column_dimensions[col_letter].width = 45

wb.save(path_results)
wb.close()

In [ ]:
if 'Volume & Coût' in wb.sheetnames: 
    del wb['Volume & Coût']
ws_vol = wb.create_sheet('Volume & Coût', 2)
ws_vol.sheet_view.showGridLines = False

ws_vol['A1'] = "VOLUME ET COÛT PAR PATHOLOGIE"
ws_vol['A1'].font = Font(bold=True, size=16, color='FFFFFF')
ws_vol['A1'].fill = PatternFill(start_color=COULEURS['bleu_fonce'], end_color=COULEURS['bleu_fonce'], fill_type='solid')
ws_vol.merge_cells('A1:E1')

entete(ws_vol['A9'], 'Pathologie', COULEURS['vert'])
entete(ws_vol['B9'], 'Volume (Ntop)', COULEURS['vert'])
entete(ws_vol['C9'], 'Coût Moyen (€)', COULEURS['vert'])
entete(ws_vol['D9'], 'Coût Total (€)', COULEURS['vert'])

row = 10
for patho in pathos:
    ws_vol[f'A{row}'] = patho
    ws_vol[f'B{row}'] = f'=IFERROR(IF(SUMIFS(cleaned_data!${c_ntop}:${c_ntop},cleaned_data!${c_patho}:${c_patho},$A{row},cleaned_data!${c_annee}:${c_annee},Dashboard!$B$3,cleaned_data!${c_dep_niv1}:${c_dep_niv1},Dashboard!$D$3,cleaned_data!${c_dep_niv2}:${c_dep_niv2},Dashboard!$F$3)>0,SUMIFS(cleaned_data!${c_ntop}:${c_ntop},cleaned_data!${c_patho}:${c_patho},$A{row},cleaned_data!${c_annee}:${c_annee},Dashboard!$B$3,cleaned_data!${c_dep_niv1}:${c_dep_niv1},Dashboard!$D$3,cleaned_data!${c_dep_niv2}:${c_dep_niv2},Dashboard!$F$3),""),"")'
    ws_vol[f'C{row}'] = f'=IFERROR(IF(B{row}="","",AVERAGEIFS(cleaned_data!${c_moy}:${c_moy},cleaned_data!${c_annee}:${c_annee},Dashboard!$B$3,cleaned_data!${c_dep_niv1}:${c_dep_niv1},Dashboard!$D$3,cleaned_data!${c_dep_niv2}:${c_dep_niv2},Dashboard!$F$3,cleaned_data!${c_patho}:${c_patho},$A{row})),"")'
    ws_vol[f'D{row}'] = f'=IFERROR(IF(B{row}="","",SUMIFS(cleaned_data!${c_montant}:${c_montant},cleaned_data!${c_annee}:${c_annee},Dashboard!$B$3,cleaned_data!${c_dep_niv1}:${c_dep_niv1},Dashboard!$D$3,cleaned_data!${c_dep_niv2}:${c_dep_niv2},Dashboard!$F$3,cleaned_data!${c_patho}:${c_patho},$A{row})),"")'
    
    ws_vol[f'A{row}'].font = Font(size=10)
    ws_vol[f'B{row}'].number_format = '#,##0'
    ws_vol[f'C{row}'].number_format = '#,##0.00'
    ws_vol[f'D{row}'].number_format = '#,##0.00'
    
    for col in ['B', 'C', 'D']:
        ws_vol[f'{col}{row}'].alignment = Alignment(horizontal='right')
    
    row += 1

dernier_row = min(row - 1, 14)
c1 = BarChart()
c1.type = "col"
c1.title = "Top Pathologies par Coût Total"
c1.height = 13
c1.width = 20
c1.add_data(Reference(ws_vol, min_col=4, min_row=9, max_row=dernier_row), titles_from_data=True)
c1.set_categories(Reference(ws_vol, min_col=1, min_row=10, max_row=dernier_row))
ws_vol.add_chart(c1, "F10")


In [160]:
if '2' in wb.sheetnames:
    del wb['2']
tdb2 = wb.create_sheet('2', 3)
tdb2.sheet_view.showGridLines = True

f_titre = Font(bold=True, color='FFFFFF', size=11)
f_texte = Font(size=10)
fill_bleu = PatternFill(start_color=COULEURS['bleu_fonce'], end_color=COULEURS['bleu_fonce'], fill_type='solid')
align_center = Alignment(horizontal='center', vertical='center')
align_left = Alignment(horizontal='left', vertical='center')
align_right = Alignment(horizontal='right', vertical='center')
trait_fin = Side(style='thin', color=COULEURS['gris_clair'])
bordure_totale = Border(left=trait_fin, right=trait_fin, top=trait_fin, bottom=trait_fin)

#  SYNTHÈSE PAR ANNÉE
tdb2['A2'] = "Année"
tdb2['B2'] = "Nombre de Patients"
tdb2['C2'] = "Coût Moyen / Patient"

for col in ['A2', 'B2', 'C2']:
    tdb2[col].font = f_titre
    tdb2[col].fill = fill_bleu
    tdb2[col].alignment = align_center

for i, annee in enumerate(annees, 2):
    row_idx = i + 1
    
    tdb2.cell(row=row_idx, column=1, value=int(annee))
    tdb2.cell(row=row_idx, column=1).alignment = align_center
    
    # Volume patients par année
    tdb2.cell(row=row_idx, column=2, value=f'=SUMIFS(cleaned_data!${c_ntop}:${c_ntop},cleaned_data!${c_annee}:${c_annee},{int(annee)})')
    tdb2.cell(row=row_idx, column=2).number_format = '#,##0'
    tdb2.cell(row=row_idx, column=2).alignment = align_right
    
    # Coût moyen par année
    tdb2.cell(row=row_idx, column=3, value=f'=AVERAGEIFS(cleaned_data!${c_moy}:${c_moy},cleaned_data!${c_annee}:${c_annee},{int(annee)})')
    tdb2.cell(row=row_idx, column=3).number_format = '#,##0.00'
    tdb2.cell(row=row_idx, column=3).alignment = align_right
    
    for col_idx in range(1, 4):
        tdb2.cell(row=row_idx, column=col_idx).font = f_texte
        tdb2.cell(row=row_idx, column=col_idx).border = bordure_totale

# TABLEAU 2 : DÉPENSES PAR POSTE
start_row_t2 = len(annees) + 6

tdb2.cell(row=start_row_t2, column=1, value="Poste de Dépense").font = f_titre
tdb2.cell(row=start_row_t2, column=1).fill = fill_bleu
tdb2.cell(row=start_row_t2, column=1).alignment = align_center

tdb2.cell(row=start_row_t2, column=2, value="Montant Global").font = f_titre
tdb2.cell(row=start_row_t2, column=2).fill = fill_bleu
tdb2.cell(row=start_row_t2, column=2).alignment = align_center
tdb2.merge_cells(start_row=start_row_t2, start_column=2, end_row=start_row_t2, end_column=3)

for i, poste in enumerate(grand_postes, 2):
    row_idx = start_row_t2 + i - 1
    
    tdb2.cell(row=row_idx, column=1, value=poste)
    tdb2.cell(row=row_idx, column=1).alignment = align_left
    
    # Montant global par poste
    tdb2.cell(row=row_idx, column=2, value=f'=SUMIFS(cleaned_data!${c_montant}:${c_montant},cleaned_data!${c_dep_niv1}:${c_dep_niv1},"{poste}")/1000000')
    tdb2.cell(row=row_idx, column=2).number_format = '#,##0.00" M€"'
    tdb2.cell(row=row_idx, column=2).alignment = align_right
    tdb2.merge_cells(start_row=row_idx, start_column=2, end_row=row_idx, end_column=3)
    
    for col_idx in range(1, 4):
        tdb2.cell(row=row_idx, column=col_idx).font = f_texte
        tdb2.cell(row=row_idx, column=col_idx).border = bordure_totale

# TABLEAU 3 : PATHOLOGIES
tdb2['E2'] = "Pathologie (Niveau 1)"
tdb2['F2'] = "Volume Patients"
tdb2['G2'] = "Coût Moyen / Patient"

for col in ['E2', 'F2', 'G2']:
    tdb2[col].font = f_titre
    tdb2[col].fill = fill_bleu
    tdb2[col].alignment = align_center

for i, patho in enumerate(pathos, 2):
    row_idx = i + 1
    
    tdb2.cell(row=row_idx, column=5, value=patho)
    tdb2.cell(row=row_idx, column=5).alignment = align_left
    
    # Volume patients par pathologie
    tdb2.cell(row=row_idx, column=6, value=f'=SUMIFS(cleaned_data!${c_ntop}:${c_ntop},cleaned_data!${c_patho}:${c_patho},"{patho}")')
    tdb2.cell(row=row_idx, column=6).number_format = '#,##0'
    tdb2.cell(row=row_idx, column=6).alignment = align_right
    
    # Coût moyen par pathologie
    tdb2.cell(row=row_idx, column=7, value=f'=AVERAGEIFS(cleaned_data!${c_moy}:${c_moy},cleaned_data!${c_patho}:${c_patho},"{patho}")')
    tdb2.cell(row=row_idx, column=7).number_format = '#,##0.00'
    tdb2.cell(row=row_idx, column=7).alignment = align_right
    
    for col_idx in range(5, 8):
        tdb2.cell(row=row_idx, column=col_idx).font = f_texte
        tdb2.cell(row=row_idx, column=col_idx).border = bordure_totale

# Largeurs colonnes
tdb2.column_dimensions['A'].width = 20
tdb2.column_dimensions['B'].width = 25
tdb2.column_dimensions['C'].width = 25
tdb2.column_dimensions['D'].width = 5
tdb2.column_dimensions['E'].width = 45
tdb2.column_dimensions['F'].width = 22
tdb2.column_dimensions['G'].width = 22


for sheet in [ws, ws_vol, tdb2]:
    sheet.column_dimensions['A'].width = 45
    for col in ['B', 'C', 'D', 'E', 'F']:
        sheet.column_dimensions[col].width = 22

wb.save(output)
wb.close()
